# Draw Reco Clusters -- DATA (Before Cosmic Tagger) -- OFF-BEAM

The OFF-BEAM counterpart of `Draw_Reco_Clusters_Data_OnBeam.ipynb` -- same
code, pointed at the off-beam (beam-off) sample instead. See that notebook's
header for the full rationale; this one only restates it briefly.

Real detector data (off-beam included) has no simulated interaction to
record: no sed-smear/sed-sce truth, no mc.json, no interaction channel, no
vertex. Every plot and every cut that depends on a true cluster is therefore
GONE, not skipped -- there is nothing to compute it from.

**What survives, reco-only:**

| kept | why |
|---|---|
| reading `img-global`, `clustering-global`, `op` (`read_data_files_for_event`, readfiles.py) | pure reco/optical, no truth file involved |
| beam-window cut (`AfterBeamWindowCut`) | bridged optical flash time, not a truth quantity |
| `RECO_ID_FIELD` grouping choice | how clustering-global's points become reco clusters, a reco-side decision |

**What is gone, and why:**

| gone | why |
|---|---|
| true-cluster stack (signal/background by channel) | needs `interaction_channel`, from mc.json |
| completeness / purity / 1-to-1 pairing | both need a true cluster to compare a reco cluster against |
| categorisation of reco clusters into signal/background bands | its categories are ALL assigned from the pairing above |
| the COSMIC TAGGER CUT | explicitly excluded here by request: DATA gets reco-level cuts only (the beam-window cut) |

**What this notebook draws and saves**

Every reco cluster -- no sampling -- that survives the beam-window cut
(`AfterBeamWindowCut`), as its own single-row XZ/YZ/XY figure
(`draw_selected_reco_cluster_view`, `AnalysisDistributions/draw_saved_clusters.py`).
Real data has no true cluster to draw beside it or to categorise it against,
so this is not a sampled illustration of a distribution (as
`Saved_Clusters/` is for the truth-bearing notebook) -- it is every selected
cluster, one figure each.

**Output layout: chunks and subchunks**

Figures are saved under `<chunk>/<subchunk>/event<n>_reco<id>.png`, mirroring
the staged input tree exactly (`input_file_name` from the staging loop is
`<chunk>__<subchunk>`, split back into the two directory levels). An index
(`selected_reco_clusters.txt`, largest reco energy first) and a
`bee_links.txt` sit at the run's top level.

**ONE BEE set for the whole run**

At the end, every event that contributed at least one drawn cluster is
assembled into ONE BEE upload (`build_population_bee_set`,
`build_bee_set_from_links.py`) -- see `feedback_bee_sets_per_run` /
`feedback_single_bee_link_per_population`: one set per run, not one per
figure or per chunk, then every figure's own per-event link inside that set
is written into `bee_links.txt`.

**Input sample and run scope**

The off-beam (beam-off) sample: `r3-beam-off-2026-09-11/bee`,
one real-data event per zip (`bee_r<run>_s<subrun>_e<event>.zip`), grouped into
`chunk_00..chunk_09`, each with `subchunk_00..subchunk_09` of 10 zips (see
`chunk_offbeam_sample.py`). `stage_nuecc_chunks()` (readfiles.py) is reused
unchanged -- it is a generic one-event-per-zip stager despite the name -- to lay
a chunk's subchunks out as `<chunk>__<subchunk>/data/<k>/` trees.

**This run is the FULL SAMPLE** (`DATA_SOURCE_CHUNKS`, `DATA_N_FILES` below):
all 10 source chunks (`chunk_00..chunk_09`), each staged whole (100 events/chunk),
1000 events total -- the same staged tree
`SignalBackground_Distributions_BeforeCosmicTagger_Data_OffBeam.ipynb` already
built and confirmed, reused as-is (idempotent staging).

**Output** -- written to
`AnalysisDistributions/multi_file_plots_charge_light_matching/Draw_Reco_Clusters_Data_OffBeam/combined_apa_<date>_<time>/`:

```
chunk_00/
  subchunk_00/
    event3_reco12.png
    ...
  subchunk_01/
    ...
...
bee_set/                     (the assembled BEE upload tree + .zip)
selected_reco_clusters.txt   (index, largest reco energy first)
bee_links.txt                (BEE SET url + one per-event url per figure)
summary.txt
```


In [ ]:
# Run scope -- same knobs as the truth-bearing notebook, kept for the same
# reason: selective filtering is useful on any sample, truth or not.
files     = "all"   # "all", or 1/2/3/... to limit the number of file subdirectories processed
events    = "all"   # "all", or 1/2/3/... to limit the number of events processed per file

# ========================================================================
# SELECTIVE FILE/EVENT FILTERING (Optional)
# ========================================================================
target_file  = None   # Set to "chunk_00__subchunk_00", etc. to process specific file only
target_event = None   # Set to a SINGLE event number (0, 1, ..., 9); use target_event_range below for a span

target_event_range = None   # e.g. (1, 5) for events 1..5
target_file_range = None    # e.g. (0, 4) for the first 5 staged units

if isinstance(target_event, (tuple, list)):
    raise ValueError(
        f"target_event={target_event} is a range, but target_event takes a single event number. "
        f"Use target_event_range={tuple(target_event)} and target_event = None instead.")

# ========================================================================
# NO LEVEL SWITCHES -- this notebook draws the JOB-LEVEL histogram only.
# ========================================================================
# Same reasoning as the truth-bearing notebook: a per-event or per-file
# histogram of one or two reco clusters is not a composition worth drawing.
# The file/event loops below still run -- they are how the clusters are
# collected -- they just do not draw or write anything of their own.


In [ ]:
%load_ext autoreload
%autoreload 2

# python libraries
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os
import time
from datetime import datetime

np.set_printoptions(linewidth=1000)

# This notebook lives in AnalysisDistributions/, one level below the
# repository root where the pipeline modules and the input trees are. Resolve
# both explicitly so the notebook runs whether Jupyter was started in this
# directory (the usual case) or at the repository root.
NB_DIR = Path.cwd()
if NB_DIR.name != "AnalysisDistributions":
    NB_DIR = NB_DIR / "AnalysisDistributions"
NB_DIR = NB_DIR.resolve()
REPO_ROOT = NB_DIR.parent

for path in (str(REPO_ROOT), str(NB_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

# Record job start time (used to report total job runtime at the end)
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Notebook directory: {NB_DIR}")
print(f"Repository root:    {REPO_ROOT}")


In [ ]:
# Pipeline modules (repository root) -- imported and used UNCHANGED. Only the
# RECO/OPTICAL side of the chain is needed here: real data has no truth file
# for anything on the true side (build_true_points_charge_light,
# cluster_category, completeness/purity, 1-to-1 pairing, mc.json vertices) to
# read, so none of that is imported.
from readfiles import stage_nuecc_chunks, read_data_files_for_event
from selections import GroupClustersByID
from metadata import build_cluster_flash_metadata, build_img_cluster_flash_metadata
from DrawRecoTrueFlashes import BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US

# Reco-energy conversion -- shared with the truth-bearing notebook's module and
# with SignalBackground_Distributions_BeforeCosmicTagger_Data_OnBeam.ipynb, so a
# cluster's reco energy means exactly the same thing everywhere.
from draw_signal_background import (
    DEFAULT_RECO_CUTS_LABEL, reco_cluster_energy_mev,
    RECO_WORK_FUNCTION_EV, RECO_RECOMBINATION_FACTOR,
)

# The per-cluster view drawer and its index/bee_links.txt writer -- the
# no-truth counterpart of draw_pair_views / write_cluster_view_index.
# categorize_reco_clusters and everything completeness/purity are NOT
# imported: there is no true cluster here to categorise a reco cluster
# against, so every SELECTED reco cluster (AfterBeamWindowCut) is drawn on its
# own, not sampled across a completeness-purity grid.
from draw_saved_clusters import draw_selected_reco_cluster_view, write_selected_reco_cluster_index

# ONE BEE set for the whole population of selected clusters' events -- see
# feedback_bee_sets_per_run / feedback_single_bee_link_per_population: build
# and upload ONE set per run, then back-fill every figure's own per-event url.
from build_bee_set_from_links import build_population_bee_set


In [ ]:
# Configuration: Parent directory containing multiple staged unit subdirectories
# (chunk_00__subchunk_00/, chunk_00__subchunk_01/, ...)
#
# Expected structure (per staged unit). stage_nuecc_chunks() below writes it.
# PARENT_DIR/
#   chunk_00__subchunk_00/data/0/0-img-global.json         (reco clusters)
#   chunk_00__subchunk_00/data/0/0-clustering-global.json  (post charge-light-matching reco clusters)
#   chunk_00__subchunk_00/data/0/0-op.json                 (optical flashes)
#   chunk_00__subchunk_00/data/1/, 2/, ... (one subdirectory per event)
#
# NO sed-smear/sed-sce truth file and NO mc.json exist anywhere in this tree --
# this is real data, not a simulated production -- so read_data_files_for_event
# (readfiles.py) does not look for them.

# ========================================================================
# INPUT SAMPLE -- the off-beam (beam-off) real-data sample: ONE event per zip
# (bee_r<run>_s<subrun>_e<event>.zip, contents at data/0/0-*.json; the "0" is
# NOT the event number, the real (run, subrun, event) is in the zip name).
# stage_nuecc_chunks() rewrites the first DATA_N_FILES zips of each source
# chunk, in groups of DATA_CHUNK_SIZE, into <chunk>__<subchunk>/data/<k>/ under
# DATA_STAGING_ROOT -- the SAME staging function the nuecc/numucc/on-beam
# samples use (it is a generic one-event-per-zip stager despite the name), so
# this loop is byte-for-byte the same shape as theirs.
# ========================================================================
SAMPLE_NAME = "OffBeam_Sample"
DATA_BEE_ROOT = Path("/Volumes/My Passport/Research_Life/Experiment/SBND/"
                     "Wirecell_Reconstruction/Samples/"
                     "r3-beam-off-2026-09-11/bee")

# FULL SAMPLE -- all 10 source chunks (chunk_00..chunk_09), each staged whole
# (all 10 subchunks x 10 files = 100 events/chunk), 1000 events total. Same
# staged tree SignalBackground_Distributions_BeforeCosmicTagger_Data_OffBeam.ipynb
# already built and confirmed -- reused as-is (idempotent staging).
DATA_SOURCE_CHUNKS = [f"chunk_{i:02d}" for i in range(10)]   # bee/chunk_NN dir(s) this job runs
DATA_STAGING_ROOT  = DATA_BEE_ROOT.parent / "staging"
DATA_N_FILES       = 100              # per source chunk (10 = subchunk_00 only; 100 = whole chunk)
DATA_CHUNK_SIZE    = 10               # events per staged subchunk

PARENT_DIR = DATA_STAGING_ROOT

# Number of files/events to process (convert the 'files'/'events' knobs above)
num_files_to_process  = None if files  == "all" else files
num_events_to_process = None if events == "all" else events

# Output directory -- the exact path requested. Unlike the energy-histogram
# notebooks, this one has no SAMPLE_NAME subdirectory beneath a shared parent:
# the user asked for this exact directory.
PLOTBASEDIR = NB_DIR / "multi_file_plots_charge_light_matching" / "Draw_Reco_Clusters_Data_OffBeam"
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"Parent directory: {PARENT_DIR}")
print(f"Plot base directory: {PLOTBASEDIR}")
print(f"Files to process: {files}")
print(f"Events to process: {events}")

if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    print(f"\nSELECTIVE FILTERING ENABLED:")
    print(f"  Target file: {target_file if target_file else 'all'}")
    print(f"  Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        print(f"  Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        print(f"  Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")

# ========================================================================
# SELECTION PARAMETERS -- RECO-LEVEL ONLY. There is no true side to select on,
# so the true-cluster cutoffs, the fiducial vertex bounds and the
# completeness/purity matching radii from the truth-bearing notebook do not
# appear here: they would have nothing to act on.
# ========================================================================

# WHICH ID FIELD defines a reco cluster in clustering-global. Same choice,
# same reasoning, as SignalBackground_Distributions_BeforeCosmicTagger.ipynb:
#   'cluster_id'      -> the COARSE grouping (everything one flash ties together)
#   'real_cluster_id' -> the FINER grouping used before 2026-08-15
RECO_ID_FIELD = 'cluster_id'

# ========================================================================
# RECO SELECTIONS -- reco-level cuts only. NO COSMIC TAGGER CUT: by request,
# DATA gets only the cuts below, not the WireCell cosmic taggers' flags.
# ========================================================================
RECO_SELECTION_NOCUTS = DEFAULT_RECO_CUTS_LABEL       # 'NoCuts': every reco cluster
RECO_SELECTION_BEAM   = 'AfterBeamWindowCut'          # flash time inside the beam window
RECO_SELECTION_LABELS = [RECO_SELECTION_NOCUTS, RECO_SELECTION_BEAM]

print("\nCuts applied: RECO-LEVEL ONLY (no truth exists to cut on)")
print(f"- reco cluster id field: {RECO_ID_FIELD}")
print(f"- beam window: {BEAM_WINDOW_MIN_US} - {BEAM_WINDOW_MAX_US} us (bridged flash time)")
print(f"- cosmic tagger cut: NOT applied (excluded by request for DATA)")
print(f"- reco energy = {RECO_WORK_FUNCTION_EV} eV * charge / {RECO_RECOMBINATION_FACTOR}")
print(f"\nReco selections counted: " + ", ".join(RECO_SELECTION_LABELS))
print(f"'Selected' (the drawn/saved population) means: {RECO_SELECTION_BEAM}")

# ========================================================================
# ONE-TIME EXTRACTION
# ========================================================================
# Idempotent -- a data/<k>/ that already holds files is left alone -- so
# re-running this notebook never re-stages an already-staged unit.
staging_start = time.time()
staged_chunks = []
for source_chunk in DATA_SOURCE_CHUNKS:
    print(f"Staging {DATA_N_FILES} zips from {DATA_BEE_ROOT / source_chunk} "
          f"in subchunks of {DATA_CHUNK_SIZE}")
    staged_chunks += stage_nuecc_chunks(
        DATA_BEE_ROOT / source_chunk, DATA_STAGING_ROOT,
        n_files=DATA_N_FILES, chunk_size=DATA_CHUNK_SIZE)
print(f"Staged {len(staged_chunks)} subchunk dir(s) from "
      f"{len(DATA_SOURCE_CHUNKS)} source chunk(s)")
staging_seconds = time.time() - staging_start


In [ ]:
def find_all_input_directories(parent_dir):
    """
    Scan parent directory for all subdirectories containing 'data' folder.
    Returns a list of file directories.
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        print(f"Error: Parent directory {parent_dir} does not exist")
        return []

    data_dirs = []
    for subdir in sorted(parent_dir.iterdir()):
        if subdir.is_dir():
            data_path = subdir / "data"
            if data_path.exists() and data_path.is_dir():
                data_dirs.append(subdir)
                print(f"Found: {subdir}")

    return data_dirs


def file_index_from_name(name):
    """
    Trailing integer of a file directory name, or None if it has no trailing
    digits. Used by target_file_range so files are selected by their real
    index rather than by position in the lexicographically sorted list.
    """
    digits = ""
    for ch in reversed(name):
        if not ch.isdigit():
            break
        digits = ch + digits
    return int(digits) if digits else None


def detect_events_in_directory(input_dir):
    """
    Auto-detect the number of events in a directory.
    Events are identified as numeric subdirectories in data/.
    Returns a sorted list of event numbers.
    """
    input_dir = Path(input_dir)
    data_dir = input_dir / "data"

    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return []

    events = []
    for item in data_dir.iterdir():
        if item.is_dir():
            try:
                events.append(int(item.name))
            except ValueError:
                pass

    return sorted(events)


# Auto-detect all input directories from parent directory
print(f"Scanning parent directory: {PARENT_DIR}")
print("-" * 60)
input_directories = find_all_input_directories(PARENT_DIR)
# staging/ can still hold units from an earlier run with a larger DATA_N_FILES
# -- keep only the ones stage_nuecc_chunks() just wrote.
staged_names = {d.name for d in staged_chunks}
input_directories = [d for d in input_directories if d.name in staged_names]
if num_files_to_process is not None:
    input_directories = input_directories[:num_files_to_process]
else:
    num_files_to_process = len(input_directories)
print("-" * 60)

print(f"\nFound {len(input_directories)} input directories with data/\n")
if input_directories:
    for input_dir in input_directories:
        detected_events = detect_events_in_directory(input_dir)
        if detected_events:
            print(f"  {input_dir.name}/data/: {len(detected_events)} events ({min(detected_events)}-{max(detected_events)})")
        else:
            print(f"  {input_dir.name}/data/: No events found")
else:
    print(f"Error: No subdirectories with 'data/' found in {PARENT_DIR}")


In [ ]:
# ============================================================================
# MAIN PROCESSING LOOP -- draw and save every SELECTED reco cluster, DATA
# ============================================================================
# Reco-only. read_data_files_for_event returns 'reco' (img-global), 'op' and
# 'clustering' (clustering-global) -- there is no 'true'/'true_clustering'/'mc'
# key to unpack, because none of those files exist for real data.
#
# Unlike SignalBackground_Distributions_BeforeCosmicTagger_Data_OffBeam.ipynb
# (which only ever needs each cluster's total charge), this notebook needs the
# actual POINT CLOUDS, so views are drawn HERE, inside the event loop -- the
# only place the point clouds exist -- exactly as save_event_cluster_views
# does in the truth-bearing notebook (draw_saved_clusters.py).

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = PLOTBASEDIR / f"combined_apa_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\n{'='*70}")
print(f"Output directory: {output_dir}")
print(f"{'='*70}\n")

n_reco_nocuts_total    = 0   # for the summary only -- not drawn
job_selected_entries   = []  # one per drawn figure: event_key, reco_cluster_id, reco_energy_mev, n_points, path
total_events_processed = 0
total_files_processed  = 0

for file_idx, input_dir in enumerate(input_directories):
    input_file_name = input_dir.name
    # input_file_name is "<chunk>__<subchunk>" (stage_nuecc_chunks' naming) --
    # split back into the two levels the figures are saved under.
    chunk_part, subchunk_part = input_file_name.split("__", 1)

    # SELECTIVE FILTERING: Skip files that don't match target_file
    if target_file is not None and input_file_name != target_file:
        print(f"Skipping {input_file_name} (target: {target_file})")
        continue

    if target_file_range is not None:
        file_idx = file_index_from_name(input_file_name)
        range_low, range_high = target_file_range
        if file_idx is None or not (range_low <= file_idx <= range_high):
            print(f"Skipping {input_file_name} (target range: file{range_low}..file{range_high})")
            continue

    print(f"\n{'='*70}")
    print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir}")
    print(f"{'='*70}")

    events_list = detect_events_in_directory(input_dir)
    if not events_list:
        print(f"No events found in {input_dir}, skipping...")
        continue

    event_low = min(events_list)
    event_high = max(events_list) + 1 if num_events_to_process is None else event_low + num_events_to_process

    print(f"Processing events {event_low} to {event_high-1}\n")
    total_files_processed += 1

    for evt in range(event_low, event_high):
        if target_event is not None and evt != target_event:
            continue
        if target_event_range is not None:
            event_range_low, event_range_high = target_event_range
            if not (event_range_low <= evt <= event_range_high):
                continue

        result = read_data_files_for_event(input_dir, evt)
        if result is None:
            print(f"  Event {evt}: could not read data, skipping")
            continue

        event_key = f"{input_file_name}_{evt}"
        op_data = result['op']

        # ------------------------------------------------------------------
        # RECO CLUSTERS. clustering-global (post charge-light matching)
        # grouped by RECO_ID_FIELD, and nothing else: no beam-window cut, no
        # fiducial cut, no minimum point count. That is what
        # RECO_CUTS_LABEL = 'NoCuts' names.
        # ------------------------------------------------------------------
        x_clu, y_clu, z_clu, id_clu, q_clu, real_id_clu = result['clustering']
        reco_ids_clu = id_clu if RECO_ID_FIELD == 'cluster_id' else real_id_clu
        predicted_points = np.column_stack((x_clu, y_clu, z_clu, reco_ids_clu, q_clu))

        # BEAM-WINDOW CUT, the second (and only other) selection. op.json
        # flashes are attached to img-global clusters and then bridged onto
        # clustering-global clusters by point charge; a reco cluster passes if
        # its bridged flash time falls in [BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US].
        # A cluster with NO bridged flash is CUT: it has no time, so it cannot
        # be shown to be in the beam window.
        event_flash_metadata_list = build_cluster_flash_metadata(
            op_data, input_file_name, evt, "Combined", event_key)
        event_img_cluster_flash_records = build_img_cluster_flash_metadata(
            result['reco'], result['clustering'], event_flash_metadata_list,
            input_file_name, evt, "Combined", event_key)
        clu_beam_window_ids = {float(r['clustering_cluster_id']) for r in event_img_cluster_flash_records
                               if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US}
        real_to_coarse = {float(r): float(c) for r, c in zip(real_id_clu, id_clu)}
        if RECO_ID_FIELD == 'cluster_id':
            clu_beam_window_ids = {real_to_coarse[r] for r in clu_beam_window_ids
                                   if r in real_to_coarse}

        if len(predicted_points) and clu_beam_window_ids:
            beam_ids_array = np.fromiter(clu_beam_window_ids, dtype=float, count=len(clu_beam_window_ids))
            predicted_points_beam = predicted_points[np.isin(predicted_points[:, 3], beam_ids_array)]
        else:
            predicted_points_beam = predicted_points[:0]

        # NO COSMIC TAGGER CUT here -- see the header. predicted_points_beam is
        # exactly the AfterBeamWindowCut selection, untouched further.

        n_reco_nocuts_total += len(GroupClustersByID(predicted_points)) if len(predicted_points) else 0
        selected_clusters = GroupClustersByID(predicted_points_beam) if len(predicted_points_beam) else {}

        # ------------------------------------------------------------------
        # DRAW + SAVE every selected reco cluster -- one figure each, into
        # <output_dir>/<chunk>/<subchunk>/. No sampling, no truth to draw
        # beside it: this is every cluster the beam-window cut selected.
        # ------------------------------------------------------------------
        fig_dir = output_dir / chunk_part / subchunk_part
        for reco_cluster_id, cluster_points in selected_clusters.items():
            cluster_points = np.asarray(cluster_points)
            n_points = len(cluster_points)
            total_charge = float(cluster_points[:, 4].sum())
            reco_energy_mev = reco_cluster_energy_mev(total_charge)
            fig_path = fig_dir / f"event{evt}_reco{reco_cluster_id:.0f}.png"
            legend_lines = [
                f"event {event_key}",
                f"reco id {reco_cluster_id:.0f}",
                f"{n_points} points",
                f"reco E {reco_energy_mev:.0f} MeV",
            ]
            draw_selected_reco_cluster_view(
                cluster_points, fig_path,
                f"Selected reco cluster -- {RECO_SELECTION_BEAM}", legend_lines)
            job_selected_entries.append({
                'event_key':        event_key,
                'reco_cluster_id':  reco_cluster_id,
                'reco_energy_mev':  reco_energy_mev,
                'n_points':         n_points,
                'path':             fig_path,
            })

        print(
            f"  Event {evt}: "
            f"reco clusters={len(GroupClustersByID(predicted_points)) if len(predicted_points) else 0} "
            f"(in beam window={len(selected_clusters)}, drawn={len(selected_clusters)})"
        )
        total_events_processed += 1

    # Per-unit status, so a long multi-chunk job shows a milestone as each
    # staged chunk__subchunk finishes.
    _src_chunk = input_file_name.split("__")[0]
    print(f"  ==> unit {file_idx + 1}/{len(input_directories)} done: {input_file_name}  "
          f"({total_events_processed} events, "
          f"{len(job_selected_entries)} selected reco cluster view(s) drawn, "
          f"{(time.time() - job_start_time) / 60:.1f} min elapsed)")
    _next = input_directories[file_idx + 1].name if file_idx + 1 < len(input_directories) else ""
    if _src_chunk and not _next.startswith(_src_chunk + "__"):
        print(f"  ===================  SOURCE CHUNK {_src_chunk} COMPLETE  "
              f"({total_events_processed} events so far)  ===================")

# ============================================================================
# JOB-LEVEL: every file and event
# ============================================================================
print(f"\n{'='*70}")
print(f"JOB SUMMARY: {total_files_processed} file(s), {total_events_processed} event(s) processed")
print(f"Total selected reco cluster views drawn: {len(job_selected_entries)}")
print(f"{'='*70}")

# ============================================================================
# ONE BEE SET for every event that contributed at least one drawn cluster --
# see feedback_bee_sets_per_run / feedback_single_bee_link_per_population.
# key_fn splits event_key ("<chunk>__<subchunk>_<evt>") back into the staged
# unit name and the local event index build_population_bee_set needs to find
# <unit>/data/<evt>/ under DATA_STAGING_ROOT.
# ============================================================================
bee_set_url = None
if job_selected_entries:
    bee_set_url = build_population_bee_set(
        job_selected_entries, DATA_STAGING_ROOT, output_dir / "bee_set",
        "Draw_Reco_Clusters_Data_OffBeam",
        key_fn=lambda e: (str(e['event_key']).rsplit('_', 1)[0],
                          int(str(e['event_key']).rsplit('_', 1)[1])))
print(f"  population BEE set: {bee_set_url or 'none / upload failed'}")

# ============================================================================
# INDEX + bee_links.txt -- one row per drawn figure, largest reco energy first.
# ============================================================================
index_path = write_selected_reco_cluster_index(
    job_selected_entries, output_dir, bee_set_url=bee_set_url,
    preamble=[
        "EVERY reco cluster -- no sampling -- that survived the beam-window cut",
        "(AfterBeamWindowCut). Real data: no true cluster exists to draw beside it",
        "or to categorise it against, so every selected cluster gets its own",
        "single-row XZ/YZ/XY figure.",
        "",
        "NO COSMIC TAGGER CUT: excluded by request for DATA (reco-level cuts only).",
    ])
print(f"  index:     {index_path}")
print(f"  bee links: {index_path.parent / 'bee_links.txt'}")
print(f"  figures:   {output_dir} (one chunk_NN/subchunk_MM/ subdirectory per staged unit)")

# ============================================================================
# JOB SUMMARY TEXT FILE
# ============================================================================
from datetime import timedelta

job_finish_dt = datetime.now()
job_runtime   = time.time() - job_start_time

summary_lines = []
summary_lines.append("=" * 80)
summary_lines.append("JOB SUMMARY -- DRAW SELECTED RECO CLUSTERS, DATA (no truth available)")
summary_lines.append("=" * 80)
summary_lines.append(f"Generated: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append("")
summary_lines.append("Configuration:")
summary_lines.append(f"Parent directory: {PARENT_DIR}")
summary_lines.append(f"Plot base directory: {PLOTBASEDIR}")
summary_lines.append(f"Files to process: {files}")
summary_lines.append(f"Events to process: {events}")
summary_lines.append("")
summary_lines.append("Input definitions:")
summary_lines.append(f"  reco cluster id field:    {RECO_ID_FIELD}"
                     f"   (clustering-global; "
                     f"{'coarse -- all activity on one flash is ONE cluster' if RECO_ID_FIELD == 'cluster_id' else 'fine -- flash-mates stay separate'})")
summary_lines.append(f"  reco cluster file:        clustering-global")
summary_lines.append(f"  true cluster file:        NONE -- real data has no truth")
if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    summary_lines.append(f"Target file: {target_file if target_file else 'all'}")
    summary_lines.append(f"Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        summary_lines.append(f"Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        summary_lines.append(f"Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")
summary_lines.append("")
summary_lines.append("Cuts (RECO-LEVEL ONLY -- there is no true side to cut on):")
summary_lines.append(f"  {RECO_SELECTION_NOCUTS}: every reco cluster; no beam-window, "
                     f"fiducial or point-count cut")
summary_lines.append(f"  {RECO_SELECTION_BEAM}: bridged flash time in "
                     f"[{BEAM_WINDOW_MIN_US}, {BEAM_WINDOW_MAX_US}] us; a cluster with no flash is cut "
                     f"-- THIS is the drawn/saved population")
summary_lines.append(f"  cosmic tagger cut:        NOT applied (excluded by request for DATA)")
summary_lines.append(f"  reco energy estimate:     {RECO_WORK_FUNCTION_EV} eV * charge / "
                     f"{RECO_RECOMBINATION_FACTOR} = {reco_cluster_energy_mev(1.0):.6g} MeV per unit charge")
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("JOB-LEVEL AGGREGATION")
summary_lines.append("=" * 80)
summary_lines.append(f"Total files processed: {total_files_processed}")
summary_lines.append(f"Total events processed: {total_events_processed}")
summary_lines.append(f"Total reco clusters, {RECO_SELECTION_NOCUTS}: {n_reco_nocuts_total}")
summary_lines.append(f"Total reco clusters, {RECO_SELECTION_BEAM} (drawn and saved): {len(job_selected_entries)}")
summary_lines.append(f"Distinct events contributing >= 1 drawn cluster: "
                     f"{len({e['event_key'] for e in job_selected_entries})}")
summary_lines.append("")
summary_lines.append(f"BEE set (all selected events, one upload): {bee_set_url or 'none / upload failed'}")
summary_lines.append(f"Index:     {index_path}")
summary_lines.append(f"BEE links: {index_path.parent / 'bee_links.txt'}")
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("JOB RUNTIME")
summary_lines.append("=" * 80)
summary_lines.append(f"Job started at:  {datetime.fromtimestamp(job_start_time).strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Job finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Total job runtime: {timedelta(seconds=int(job_runtime))} ({job_runtime:.1f} seconds)")
summary_lines.append(f"  of which one-time input staging: {timedelta(seconds=int(staging_seconds))} ({staging_seconds:.1f} s)   -- ~0 on a re-run")
summary_lines.append("=" * 80)

with open(output_dir / "summary.txt", "w") as f:
    f.write("\n".join(summary_lines) + "\n")

print(f"\nJob finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')} (runtime: {job_runtime:.1f}s)")
print(f"Job summary written to: {output_dir / 'summary.txt'}")
